In [ ]:
# %% [markdown]
# # RQ2 — Step 0: Clone repositories
# Reads a CSV with column `repo_url`, clones/fetches into CLONE_ROOT,
# and writes a manifest with basic metadata.

# %%
from __future__ import annotations
import csv, subprocess, sys, json, time
from pathlib import Path
from typing import Optional, List
from pathlib import Path

# -----------------------------
# Config (edit as needed)
# -----------------------------


# Use a raw string r"..." for Windows paths with spaces
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
URL_LIST_CSV = WORK_ROOT / "URL_List.csv"   # put your CSV here
CLONE_ROOT   = WORK_ROOT / "clones"         # repos will clone here
MANIFEST_CSV = WORK_ROOT / "clones_manifest.csv"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)


# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# %%
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=True, text=True)

def repo_dir_name_from_url(url: str) -> str:
    # e.g. https://github.com/owner/name(.git) -> owner__name
    base = url.split("//")[-1]
    parts = base.split("/")
    if len(parts) >= 3:
        owner = parts[-2]
        name  = parts[-1].replace(".git", "")
        return f"{owner}__{name}"
    return base.replace("/", "__").replace(".git", "")

def ensure_cloned(url: str, dest_root: Path) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)
    if d.exists() and (d / ".git").exists():
        # Refresh remote info (best-effort)
        try:
            sh(["git", "fetch", "--all", "--tags", "--prune"], cwd=d)
        except Exception:
            pass
        return d
    sh(["git", "clone", "--no-tags", "--filter=blob:none", "--recurse-submodules=no", url, str(d)])
    return d

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir)
    return int(cp.stdout.strip() or "0")

# %%
assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

rows, ok, fail = [], 0, 0
with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        url = (row.get("repo_url") or "").strip()
        if not url:
            continue
        t0 = time.time()
        rec = {"repo_url": url, "dir": None, "status": "unknown", "seconds": None, "total_commits": None, "error": ""}
        try:
            d = ensure_cloned(url, CLONE_ROOT)
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)
            rec["status"] = "ok"
            ok += 1
        except subprocess.CalledProcessError as e:
            rec["status"] = "error"
            rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
            fail += 1
        rec["seconds"] = round(time.time() - t0, 2)
        rows.append(rec)
        print(f"[{rec['status']}] {url} -> {rec['dir']} ({rec['seconds']}s)")

# %%
# Write manifest
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["repo_url","dir","status","seconds","total_commits","error"])
    w.writeheader()
    w.writerows(rows)

print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


[ok] https://github.com/connectbot/connectbot -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\connectbot__connectbot (1.64s)
[ok] https://github.com/robolectric/robolectric -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\robolectric__robolectric (7.22s)
[ok] https://github.com/opendocument-app/OpenDocument.droid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\opendocument-app__OpenDocument.droid (4.29s)
[ok] https://github.com/maxpower47/PinDroid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\maxpower47__PinDroid (2.95s)
[ok] https://github.com/Rajawali/Rajawali -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\Rajawali__Rajawali (6.46s)
[ok] https://github.com/cgeo/cgeo -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\cgeo__cgeo (24.03s)
[ok] https://github.com/OneBusAway/onebusaway-android -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\OneBusAway__onebus

In [ ]:
# %% [markdown]
# # RQ2 — Step 1: Mine commit snapshots
# Scans cloned repos for commits touching CI/YAML/Gradle/scripts and writes per-repo JSONL snapshots.

# %%
# Optional (uncomment): install YAML support for richer parsing
# !pip install pyyaml -q

# %%
from __future__ import annotations
import re, json, subprocess, csv
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

# -----------------------------
# Config (edit as needed)
# -----------------------------
from pathlib import Path

WORK_ROOT      = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
CLONE_ROOT     = WORK_ROOT / "clones"
SNAPSHOT_DIR   = WORK_ROOT / "snapshots"   # per-repo JSONL output here
MAX_COMMITS_PER_REPO = 0                   # 0 = no limit

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)


SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# %%
try:
    import yaml  # optional; if missing we fall back to regex scans
except Exception:
    yaml = None

def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=True, text=True)

# Relevant file surfaces
CI_FILES   = [".gitlab-ci.yml", ".circleci/config.yml", "bitrise.yml", "azure-pipelines.yml"]
GRADLE_FILES = ["build.gradle", "build.gradle.kts", "settings.gradle", "settings.gradle.kts", "gradle.properties", "gradle/wrapper/gradle-wrapper.properties"]

def is_ci_file(p: str) -> bool:
    return p.startswith(".github/workflows/") or p in CI_FILES

def is_gradle_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(x) for x in (f.lower() for f in GRADLE_FILES))

def is_script_file(p: str) -> bool:
    lp = p.lower()
    return lp.endswith(".sh") or lp.endswith(".py") or ("script" in lp)

def touched_relevant(paths: List[str]) -> bool:
    for p in paths:
        if not p.strip():
            continue
        if is_ci_file(p) or is_gradle_file(p) or is_script_file(p):
            return True
    return False

def list_relevant_commits(repo_dir: Path) -> List[Tuple[str, int, List[str]]]:
    # git log --name-only --pretty=%H\t%ct
    cp = sh(["git", "log", "--all", "--name-only", "--pretty=%H%x09%ct"], cwd=repo_dir)
    results: List[Tuple[str, int, List[str]]] = []
    sha, ts = None, None
    changed: List[str] = []
    for line in cp.stdout.splitlines():
        if re.match(r"^[0-9a-f]{40}\t\d+$", line):
            if sha is not None and touched_relevant(changed):
                results.append((sha, ts, changed))
            sha, ts_s = line.split("\t", 1)
            ts = int(ts_s)
            changed = []
        else:
            if line.strip():
                changed.append(line.strip())
    if sha is not None and touched_relevant(changed):
        results.append((sha, ts, changed))
    results.reverse()  # oldest -> newest
    return results

def git_show(repo_dir: Path, sha: str, path: str) -> Optional[str]:
    try:
        cp = sh(["git", "show", f"{sha}:{path}"], cwd=repo_dir)
        return cp.stdout
    except subprocess.CalledProcessError:
        return None

def git_subject(repo_dir: Path, sha: str) -> str:
    cp = sh(["git", "log", "-1", "--pretty=%s", sha], cwd=repo_dir)
    return cp.stdout.strip()

# Heuristic extractors
RE_INT      = re.compile(r"\d+")
RE_VERSION  = re.compile(r"\b(\d+(?:\.\d+){0,3})\b")

YAML_KEYS = {
    "api": ["api-level","apilevel","api_level"],
    "abi": ["abi","arch","cpu","abi_filters","abi-filter"],
    "system_image": ["system-image","target","systemimage"],
    "device": ["device","avd-name","avd","device-profile","model","hardwareProfile"],
    "orchestrator": ["orchestrator","android-test-orchestrator","use-orchestrator"],
    "wait": ["wait-for-boot","wait_for_boot"],
    "timeouts": ["emulator-boot-timeout","timeout","test-timeout","emulator_timeout"],
    "retries": ["retry","retries","max-retries"],
    "matrix": ["matrix","strategy"],
    "runner_os": ["runs-on","machine","image"],
    "jdk": ["java-version","jdk","java","distribution"],
    "invocation": ["run","gradle_args","gradlew_args","task","tasks"],
    "thirdparty": ["browserstack","saucelabs","firebase","bitbar","kobiton","testlab","devicefarm"]
}

def extract_from_yaml_text(text: str) -> Dict[str, Any]:
    if yaml is None:
        return {}
    try:
        docs = list(yaml.safe_load_all(text))
    except Exception:
        docs = []
    out = {
        "api_levels": set(), "abis": set(), "system_images": set(), "device_profiles": set(),
        "orchestrator": None, "wait_for_boot": None, "timeouts": {}, "retries": None,
        "matrix_axes": set(), "runner_os": None, "jdk": None, "invocation_hints": [], "thirdparty_refs": set(),
    }
    def scan_obj(obj):
        if isinstance(obj, dict):
            for k,v in obj.items():
                lk = str(k).lower()
                if lk in (x.lower() for x in YAML_KEYS["api"]):
                    if isinstance(v, list):
                        for x in v:
                            if isinstance(x, (int,str)) and RE_INT.search(str(x)):
                                out["api_levels"].add(int(RE_INT.search(str(x)).group()))
                    elif isinstance(v, (int,str)):
                        m = RE_INT.search(str(v))
                        if m: out["api_levels"].add(int(m.group()))
                if lk in (x.lower() for x in YAML_KEYS["abi"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): out["abis"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["system_image"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): out["system_images"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["device"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): out["device_profiles"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["orchestrator"]):
                    if isinstance(v, bool): out["orchestrator"] = v
                    elif isinstance(v, str): out["orchestrator"] = v.lower() in ("1","true","yes","on")
                if lk in (x.lower() for x in YAML_KEYS["wait"]):
                    if isinstance(v, bool): out["wait_for_boot"] = v
                    elif isinstance(v, str): out["wait_for_boot"] = v.lower() in ("1","true","yes","on")
                if lk in (x.lower() for x in YAML_KEYS["timeouts"]):
                    out["timeouts"][k] = v
                if lk in (x.lower() for x in YAML_KEYS["retries"]):
                    try: out["retries"] = int(RE_INT.search(str(v)).group())
                    except Exception: pass
                if lk in (x.lower() for x in YAML_KEYS["matrix"]):
                    if isinstance(v, dict):
                        for ax, vals in v.items():
                            out["matrix_axes"].add(str(ax))
                if lk in (x.lower() for x in YAML_KEYS["runner_os"]):
                    out["runner_os"] = str(v)
                if lk in (x.lower() for x in YAML_KEYS["jdk"]):
                    out["jdk"] = str(v)
                if lk in (x.lower() for x in YAML_KEYS["invocation"]):
                    out["invocation_hints"].append(str(v))
                if isinstance(v, (dict, list)): scan_obj(v)
        elif isinstance(obj, list):
            for x in obj: scan_obj(x)

    for d in docs:
        scan_obj(d)
    # convert sets to sorted lists for JSON
    out["api_levels"]     = sorted(out["api_levels"])
    out["abis"]           = sorted(out["abis"])
    out["system_images"]  = sorted(out["system_images"])
    out["device_profiles"]= sorted(out["device_profiles"])
    out["matrix_axes"]    = sorted(out["matrix_axes"])
    out["thirdparty_refs"]= sorted(out["thirdparty_refs"])
    return out

def extract_from_text(path: str, text: str) -> Dict[str, Any]:
    if path.lower().endswith((".yml",".yaml")) and yaml is not None:
        data = extract_from_yaml_text(text)
    else:
        data = {}
    # Gradle heuristics
    if path.endswith(("build.gradle","build.gradle.kts","gradle.properties","gradle/wrapper/gradle-wrapper.properties","settings.gradle","settings.gradle.kts")):
        agp = re.findall(r"com\.android\.tools\.build:gradle:([0-9][^'\"\s]+)", text)
        if agp:
            data["agp_versions"] = sorted(set(agp))
        for m in re.finditer(r"apiLevel\s*=\s*(\d+)", text, re.IGNORECASE):
            data.setdefault("api_levels", [])
            if int(m.group(1)) not in data["api_levels"]:
                data["api_levels"].append(int(m.group(1)))
        if re.search(r"ANDROIDX_TEST_ORCHESTRATOR", text):
            data["orchestrator"] = True
    return data

def write_jsonl(path: Path, rows: List[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# %%
repos = [p for p in CLONE_ROOT.iterdir() if (p / ".git").exists()]
print(f"Found {len(repos)} repos in {CLONE_ROOT}")

for repo in repos:
    rel_commits = list_relevant_commits(repo)
    if MAX_COMMITS_PER_REPO > 0:
        rel_commits = rel_commits[:MAX_COMMITS_PER_REPO]
    out_rows: List[dict] = []
    for sha, ts, changed_paths in rel_commits:
        rel_paths = [p for p in changed_paths if is_ci_file(p) or is_gradle_file(p) or is_script_file(p)]
        if not rel_paths: 
            continue
        subj = git_subject(repo, sha)
        for pth in rel_paths:
            txt = git_show(repo, sha, pth)
            if txt is None:
                continue
            feats = extract_from_text(pth, txt)
            out_rows.append({
                "repo": repo.name,
                "sha": sha,
                "timestamp": ts,
                "subject": subj,
                "path": pth,
                "features": feats,
            })
    if not out_rows:
        print(f"[skip] {repo.name}: no relevant snapshots")
        continue
    dst = SNAPSHOT_DIR / f"{repo.name}.jsonl"
    write_jsonl(dst, out_rows)
    print(f"[ok] {repo.name}: {len(out_rows)} snapshots -> {dst}")


In [ ]:
# %% [markdown]
# # RQ2 — Step 2: Build episodes (CCEs) + labels
# Groups commit snapshots into time-based episodes and assigns labels: Fix / Upgrade / Enhancement.

# %%
from __future__ import annotations
import json
from pathlib import Path
from typing import List, Dict, Any

# -----------------------------
# Config (edit as needed)
# -----------------------------
from pathlib import Path

WORK_ROOT        = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR     = WORK_ROOT / "snapshots"
CCE_DIR          = WORK_ROOT / "cce"
EPISODE_GAP_HOURS = 24  # new episode if gap > 24h

CCE_DIR.mkdir(parents=True, exist_ok=True)


CCE_DIR.mkdir(parents=True, exist_ok=True)

# %%
def read_jsonl(path: Path) -> List[dict]:
    if not path.exists(): return []
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def label_episode(commits: List[dict]) -> Dict[str, Any]:
    msg = " ".join((c.get("subject") or "").lower() for c in commits)
    any_fix_words = any(w in msg for w in ["fix","bug","hotfix","regression","flaky"])
    any_upgrade_words = any(w in msg for w in ["upgrade","bump","update","migrate"])

    def max_api(c): 
        apis = (c.get("features", {}).get("api_levels") or [])
        return max(apis) if apis else None
    apis = [max_api(c) for c in commits if max_api(c) is not None]
    api_increase = (len(apis) >= 2 and max(apis) > min(apis))

    agp_versions = []
    for c in commits:
        agp_versions += (c.get("features",{}).get("agp_versions") or [])
    agp_versions = sorted(set(agp_versions))
    agp_upgrade = len(agp_versions) > 1

    orchestrator_on = any(bool(c.get("features",{}).get("orchestrator")) for c in commits)

    reasons = []
    if api_increase: reasons.append("api_level_increase")
    if agp_upgrade:  reasons.append("agp_upgrade")
    if orchestrator_on: reasons.append("orchestrator_enabled")
    if any_upgrade_words: reasons.append("upgrade_keyword")
    if any_fix_words: reasons.append("fix_keyword")

    if api_increase or agp_upgrade or any_upgrade_words:
        label = "Upgrade"
    elif any_fix_words:
        label = "Fix"
    else:
        label = "Enhancement"

    return {"label": label, "reasons": reasons}

def group_into_episodes(rows: List[dict], gap_hours: int) -> List[List[dict]]:
    if not rows: return []
    rows = sorted(rows, key=lambda r: r["timestamp"])
    episodes: List[List[dict]] = [[rows[0]]]
    gap = gap_hours * 3600
    for prev, cur in zip(rows, rows[1:]):
        if cur["timestamp"] - prev["timestamp"] > gap:
            episodes.append([cur])
        else:
            episodes[-1].append(cur)
    return episodes

# %%
per_repo = {}
for p in SNAPSHOT_DIR.glob("*.jsonl"):
    repo = p.stem
    rows = read_jsonl(p)
    if not rows:
        continue
    episodes = group_into_episodes(rows, EPISODE_GAP_HOURS)
    out_rows = []
    for idx, ep in enumerate(episodes, start=1):
        lab = label_episode(ep)
        start, end = ep[0], ep[-1]
        out_rows.append({
            "repo": repo,
            "episode_id": f"{repo}__{idx:04d}",
            "start_sha": start["sha"],
            "end_sha": end["sha"],
            "start_ts": start["timestamp"],
            "end_ts": end["timestamp"],
            "num_commits": len(ep),
            "label": lab["label"],
            "reasons": lab["reasons"],
        })
    dst = CCE_DIR / f"{repo}.jsonl"
    with dst.open("w", encoding="utf-8") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"[ok] {repo}: {len(out_rows)} episodes -> {dst}")

print("Done.")


In [ ]:
# %% [markdown]
# # RQ2 — Step 3: Combine per-repo JSONLs into master CSV/Parquet
# Produces combined CSVs for snapshots and episodes. Parquet if pandas/pyarrow are installed.

# %%
# Optional (uncomment): install for Parquet output
# !pip install pandas pyarrow -q

# %%
from __future__ import annotations
import json, csv
from pathlib import Path
from typing import List

# -----------------------------
# Config (edit as needed)
# -----------------------------
from pathlib import Path

WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR = WORK_ROOT / "snapshots"
CCE_DIR      = WORK_ROOT / "cce"
COMBINE_DIR  = WORK_ROOT / "combined"

COMBINE_DIR.mkdir(parents=True, exist_ok=True)


# %%
def read_all_jsonl(folder: Path) -> List[dict]:
    out = []
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    d = json.loads(line)
                    d["_source_file"] = p.name
                    out.append(d)
    return out

snapshots = read_all_jsonl(SNAPSHOT_DIR)
episodes  = read_all_jsonl(CCE_DIR)

# %%
# Write CSV (snapshots)
snap_csv = COMBINE_DIR / "snapshots_combined.csv"
if snapshots:
    keys = sorted(set().union(*[set(x.keys()) for x in snapshots]))
    with snap_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in snapshots:
            # flatten features minimally
            feats = r.get("features", {})
            r = {**r, 
                 "features_json": json.dumps(feats, ensure_ascii=False)}
            w.writerow(r)
    print(f"[ok] {snap_csv}")
else:
    print("[warn] No snapshots found.")

# Write CSV (episodes)
cce_csv = COMBINE_DIR / "episodes_combined.csv"
if episodes:
    keys = sorted(set().union(*[set(x.keys()) for x in episodes]))
    with cce_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in episodes:
            w.writerow(r)
    print(f"[ok] {cce_csv}")
else:
    print("[warn] No episodes found.")

# %%
# Optional: Parquet with pandas (if available)
try:
    import pandas as pd
    if snapshots:
        import pandas as pd
        df_s = pd.DataFrame(snapshots)
        df_s.to_parquet(COMBINE_DIR / "snapshots_combined.parquet", index=False)
        print(f"[ok] Parquet: snapshots_combined.parquet")
    if episodes:
        df_e = pd.DataFrame(episodes)
        df_e.to_parquet(COMBINE_DIR / "episodes_combined.parquet", index=False)
        print(f"[ok] Parquet: episodes_combined.parquet")
except Exception as e:
    print(f"[note] Skipping Parquet (pandas/pyarrow not available?): {e}")
